In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
Link Prediction Benchmark
=========================
This script benchmarks four link prediction methods on a directed graph built
from an XML file (e.g., oncogene_pathways.xml):

1. Adamic‑Adar + Jaccard (combined score)
2. Node2Vec (undirected) + Random Forest
3. Logistic Regression on topological features (CN, AA, Jaccard, PA)
4. Node2Vec (directed) + Random Forest

Metrics: AUC-ROC, Average Precision (AP), Precision@50, Precision@100.
Top predictions are validated against the STRING database.

Usage:
    python 10_link_prediction_benchmark.py
"""

import os
import random
import time
import xml.etree.ElementTree as ET
from collections import defaultdict

import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from node2vec import Node2Vec
from gensim.models import Word2Vec
import requests

# Set random seeds for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

# ----------------------------------------------------------------------
# 1. Build graph from XML
# ----------------------------------------------------------------------
def build_graph_from_xml(xml_file):
    """Parse XML and return a directed graph."""
    G = nx.DiGraph()
    tree = ET.parse(xml_file)
    root = tree.getroot()
    for pathway in root.findall('pathway'):
        node_list = pathway.find('nodeList')
        if node_list is not None:
            for node in node_list.findall('node'):
                node_name = node.get('name')
                if node_name:
                    G.add_node(node_name)
        edge_list = pathway.find('edgeList')
        if edge_list is not None:
            for edge in edge_list.findall('edge'):
                start = edge.find('startNode').get('node')
                end = edge.find('endNode').get('node')
                if start and end:
                    G.add_edge(start, end)
    return G

# ----------------------------------------------------------------------
# 2. Degree‑matched negative sampling
# ----------------------------------------------------------------------
def degree_matched_negative_edges(positive_edges, G, n_negatives_per_positive=1):
    """For each positive edge (u,v), sample a negative edge (u,w) where w is not v,
       (u,w) does not exist, and out-degree of w is close to that of v."""
    negative_edges = []
    all_nodes = list(G.nodes())
    out_deg = {n: G.out_degree(n) for n in all_nodes}
    for u, v in positive_edges:
        target_deg = out_deg[v]
        # Candidate w: not v, no edge (u,w), degree within 20% of target
        candidates = [w for w in all_nodes
                      if w != v and not G.has_edge(u, w) and abs(out_deg[w] - target_deg) <= max(1, target_deg * 0.2)]
        if not candidates:
            # Fallback: any non‑neighbor
            candidates = [w for w in all_nodes if w != v and not G.has_edge(u, w)]
        w = np.random.choice(candidates)
        negative_edges.append((u, w))
    return negative_edges

# ----------------------------------------------------------------------
# 3. Evaluation helper
# ----------------------------------------------------------------------
def evaluate_predictions(y_true, y_score, method_name, top_k=[50, 100]):
    """Compute AUC-ROC, Average Precision, and Precision@K."""
    auc = roc_auc_score(y_true, y_score)
    ap = average_precision_score(y_true, y_score)
    results = {'Method': method_name, 'AUC-ROC': auc, 'AP': ap}
    for k in top_k:
        top_k_idx = np.argsort(y_score)[-k:]
        y_pred_topk = np.zeros_like(y_true)
        y_pred_topk[top_k_idx] = 1
        precision_k = precision_score(y_true, y_pred_topk, zero_division=0)
        results[f'Precision@{k}'] = precision_k
    return results

def string_validation(pred_edges, top_k=50, delay=0.5):
    """Query STRING database for top_k predicted edges and return fraction known."""
    def check_string(u, v):
        url = f"https://string-db.org/api/json/network?identifiers={u}%0D{v}&species=9606"
        try:
            resp = requests.get(url, timeout=5)
            if resp.status_code == 200:
                data = resp.json()
                return len(data) > 0
        except Exception:
            pass
        return False

    known = 0
    for i, (u, v) in enumerate(pred_edges[:top_k]):
        if check_string(u, v):
            known += 1
        if i % 10 == 0:
            time.sleep(delay)
    return known / top_k if top_k > 0 else 0

# ----------------------------------------------------------------------
# 4. Method 1: Adamic‑Adar + Jaccard (combined)
# ----------------------------------------------------------------------
def adamic_adar_jaccard_score(G, u, v):
    """Normalized sum of Adamic‑Adar and Jaccard (undirected)."""
    G_und = G.to_undirected()
    aa = list(nx.adamic_adar_index(G_und, [(u, v)]))[0][2]
    jc = list(nx.jaccard_coefficient(G_und, [(u, v)]))[0][2]
    return aa + jc

# ----------------------------------------------------------------------
# 5. Method 2: Node2Vec (undirected) + Random Forest
# ----------------------------------------------------------------------
def node2vec_undirected_embeddings(G, dimensions=64, walk_length=30, num_walks=200, workers=4):
    G_und = G.to_undirected()
    node2vec = Node2Vec(G_und, dimensions=dimensions, walk_length=walk_length,
                        num_walks=num_walks, workers=workers, seed=RANDOM_STATE)
    model = node2vec.fit(window=10, min_count=1, batch_words=4)
    return {node: model.wv[node] for node in G_und.nodes()}

def hadamard_product(emb, u, v):
    return emb[u] * emb[v]

# ----------------------------------------------------------------------
# 6. Method 3: Logistic Regression on topological features
# ----------------------------------------------------------------------
def topological_features(G, u, v):
    G_und = G.to_undirected()
    cn = len(list(nx.common_neighbors(G_und, u, v)))
    aa = list(nx.adamic_adar_index(G_und, [(u, v)]))[0][2] if cn > 0 else 0
    jc = list(nx.jaccard_coefficient(G_und, [(u, v)]))[0][2] if cn > 0 else 0
    pa = G.out_degree(u) * G.in_degree(v)
    return [cn, aa, jc, pa]

# ----------------------------------------------------------------------
# 7. Method 4: Node2Vec (directed) + Random Forest
# ----------------------------------------------------------------------
def directed_random_walk(G, start, walk_length):
    walk = [start]
    for _ in range(walk_length - 1):
        curr = walk[-1]
        nbrs = list(G.successors(curr))
        if not nbrs:
            break
        next_node = np.random.choice(nbrs)
        walk.append(next_node)
    return [str(x) for x in walk]

def directed_node2vec_embeddings(G, num_walks=200, walk_length=30, dimensions=64, window=10, workers=4):
    walks = []
    nodes = list(G.nodes())
    for node in nodes:
        for _ in range(num_walks):
            walk = directed_random_walk(G, node, walk_length)
            walks.append(walk)
    model = Word2Vec(walks, vector_size=dimensions, window=window, min_count=1, sg=1, workers=workers, seed=RANDOM_STATE)
    emb = {}
    for node in nodes:
        key = str(node)
        if key in model.wv:
            emb[node] = model.wv[key]
        else:
            emb[node] = np.zeros(dimensions)
    return emb

def edge_feature_concat(emb, u, v):
    return np.concatenate([emb[u], emb[v]])

# ----------------------------------------------------------------------
# 8. Main pipeline
# ----------------------------------------------------------------------
def main():
    xml_file = "oncogene_pathways.xml"
    if not os.path.exists(xml_file):
        raise FileNotFoundError(f"Input file {xml_file} not found. Please run previous steps first.")

    # Build graph
    G = build_graph_from_xml(xml_file)
    print(f"Directed graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

    # Split edges
    edges = list(G.edges())
    train_edges, test_edges = train_test_split(edges, test_size=0.2, random_state=RANDOM_STATE)
    print(f"Training positive edges: {len(train_edges)}, Test positive edges: {len(test_edges)}")

    # Generate negative edges
    train_neg = degree_matched_negative_edges(train_edges, G)
    test_neg = degree_matched_negative_edges(test_edges, G)
    print(f"Training negative edges: {len(train_neg)}, Test negative edges: {len(test_neg)}")

    # Build label arrays
    X_train_edges = train_edges + train_neg
    y_train = [1] * len(train_edges) + [0] * len(train_neg)
    X_test_edges = test_edges + test_neg
    y_test = [1] * len(test_edges) + [0] * len(test_neg)
    print(f"Training set size: {len(X_train_edges)}, Test set size: {len(X_test_edges)}")

    # ------------------- Method 1: Adamic-Adar + Jaccard -------------------
    print("\n--- Method 1: Adamic-Adar + Jaccard ---")
    y_score_aa_jc = [adamic_adar_jaccard_score(G, u, v) for u, v in X_test_edges]
    results_aa_jc = evaluate_predictions(y_test, y_score_aa_jc, "Adamic-Adar + Jaccard")
    # STRING validation on top 50 predictions
    test_scores = list(zip(X_test_edges, y_score_aa_jc))
    test_scores.sort(key=lambda x: x[1], reverse=True)
    top50_edges = [e for e, _ in test_scores[:50]]
    string_valid_aa_jc = string_validation(top50_edges, top_k=50)
    results_aa_jc['STRING Validated @50'] = string_valid_aa_jc
    print(results_aa_jc)

    # ------------------- Method 2: Node2Vec Undirected + RF -------------------
    print("\n--- Method 2: Node2Vec Undirected + RF ---")
    emb_und = node2vec_undirected_embeddings(G)
    X_train_feat = np.array([hadamard_product(emb_und, u, v) for u, v in X_train_edges])
    X_test_feat = np.array([hadamard_product(emb_und, u, v) for u, v in X_test_edges])
    rf_und = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
    rf_und.fit(X_train_feat, y_train)
    y_score_rf_und = rf_und.predict_proba(X_test_feat)[:, 1]
    results_rf_und = evaluate_predictions(y_test, y_score_rf_und, "Node2Vec Undirected + RF")
    test_scores = list(zip(X_test_edges, y_score_rf_und))
    test_scores.sort(key=lambda x: x[1], reverse=True)
    top50_edges = [e for e, _ in test_scores[:50]]
    string_valid_rf_und = string_validation(top50_edges, top_k=50)
    results_rf_und['STRING Validated @50'] = string_valid_rf_und
    print(results_rf_und)

    # ------------------- Method 3: Logistic Regression -------------------
    print("\n--- Method 3: Logistic Regression ---")
    X_train_topo = np.array([topological_features(G, u, v) for u, v in X_train_edges])
    X_test_topo = np.array([topological_features(G, u, v) for u, v in X_test_edges])
    scaler = StandardScaler()
    X_train_topo = scaler.fit_transform(X_train_topo)
    X_test_topo = scaler.transform(X_test_topo)
    lr = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
    lr.fit(X_train_topo, y_train)
    y_score_lr = lr.predict_proba(X_test_topo)[:, 1]
    results_lr = evaluate_predictions(y_test, y_score_lr, "Logistic Regression")
    test_scores = list(zip(X_test_edges, y_score_lr))
    test_scores.sort(key=lambda x: x[1], reverse=True)
    top50_edges = [e for e, _ in test_scores[:50]]
    string_valid_lr = string_validation(top50_edges, top_k=50)
    results_lr['STRING Validated @50'] = string_valid_lr
    print(results_lr)

    # ------------------- Method 4: Node2Vec Directed + RF -------------------
    print("\n--- Method 4: Node2Vec Directed + RF ---")
    emb_dir = directed_node2vec_embeddings(G)
    X_train_dir = np.array([edge_feature_concat(emb_dir, u, v) for u, v in X_train_edges])
    X_test_dir = np.array([edge_feature_concat(emb_dir, u, v) for u, v in X_test_edges])
    rf_dir = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
    rf_dir.fit(X_train_dir, y_train)
    y_score_rf_dir = rf_dir.predict_proba(X_test_dir)[:, 1]
    results_rf_dir = evaluate_predictions(y_test, y_score_rf_dir, "Node2Vec Directed + RF")
    test_scores = list(zip(X_test_edges, y_score_rf_dir))
    test_scores.sort(key=lambda x: x[1], reverse=True)
    top50_edges = [e for e, _ in test_scores[:50]]
    string_valid_rf_dir = string_validation(top50_edges, top_k=50)
    results_rf_dir['STRING Validated @50'] = string_valid_rf_dir
    print(results_rf_dir)

    # ------------------- Summary table -------------------
    all_results = [results_aa_jc, results_rf_und, results_lr, results_rf_dir]
    df_results = pd.DataFrame(all_results)
    cols = ['Method', 'AUC-ROC', 'AP', 'Precision@50', 'Precision@100', 'STRING Validated @50']
    df_results = df_results[cols]
    print("\n========== Link Prediction Method Comparison ==========")
    print(df_results.to_string(index=False))

    # ------------------- ROC curves -------------------
    plt.figure(figsize=(8, 6))
    methods = [
        ("Adamic-Adar + Jaccard", y_score_aa_jc),
        ("Node2Vec Undirected + RF", y_score_rf_und),
        ("Logistic Regression", y_score_lr),
        ("Node2Vec Directed + RF", y_score_rf_dir)
    ]
    for name, y_score in methods:
        fpr, tpr, _ = roc_curve(y_test, y_score)
        auc = roc_auc_score(y_test, y_score)
        plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curves for Link Prediction Methods")
    plt.legend(loc="lower right")
    plt.grid(True)
    plt.savefig("link_prediction_roc.png", dpi=300)
    plt.show()

    print("\nBenchmark completed. ROC curve saved as link_prediction_roc.png")

if __name__ == "__main__":
    main()